In [1]:
import sys
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

print(sys.executable)

c:\Users\Lenovo\Desktop\projects_GitHub\fitness-calories-eda\venv\Scripts\python.exe


In [2]:
exercise = pd.read_csv("../data/raw/exercise.csv")
exercise.head()

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8


In [3]:

exercise.shape


(15000, 8)

In [4]:
exercise.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   User_ID     15000 non-null  int64  
 1   Gender      15000 non-null  str    
 2   Age         15000 non-null  int64  
 3   Height      15000 non-null  float64
 4   Weight      15000 non-null  float64
 5   Duration    15000 non-null  float64
 6   Heart_Rate  15000 non-null  float64
 7   Body_Temp   15000 non-null  float64
dtypes: float64(5), int64(2), str(1)
memory usage: 937.6 KB


In [5]:
exercise.describe()

,User_ID,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
count,1.500000e+04,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,1.497736e+07,42.789800,174.465133,74.966867,15.530600,95.518533,40.025453
std,2.872851e+06,16.980264,14.258114,15.035657,8.319203,9.583328,0.779230
min,1.000116e+07,20.000000,123.000000,36.000000,1.000000,67.000000,37.100000
25%,1.247419e+07,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000
50%,1.499728e+07,39.000000,175.000000,74.000000,16.000000,96.000000,40.200000
75%,1.744928e+07,56.000000,185.000000,87.000000,23.000000,103.000000,40.600000
max,1.999965e+07,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000


In [6]:
calories = pd.read_csv("../data/raw/calories.csv")
calories.head()

,User_ID,Calories
0,14733363,231.0
1,14861698,66.0
2,11179863,26.0
3,16180408,71.0
4,17771927,35.0


In [7]:
calories.shape

(15000, 2)

In [8]:
calories.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   User_ID   15000 non-null  int64  
 1   Calories  15000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 234.5 KB


In [9]:
calories.describe()

,User_ID,Calories
count,1.500000e+04,15000.000000
mean,1.497736e+07,89.539533
std,2.872851e+06,62.456978
min,1.000116e+07,1.000000
25%,1.247419e+07,35.000000
50%,1.499728e+07,79.000000
75%,1.744928e+07,138.000000
max,1.999965e+07,314.000000


## Initial Data Overview

**Exercise dataset (15000 × 8):**
Columns — User_ID (id), Gender (cat), Age (years), Height (cm),
Weight (kg), Duration (min), Heart_Rate (bpm), Body_Temp (°C).

**Calories dataset (15000 × 2):**
Columns — User_ID (id), Calories (kcal).

**Ranges look reasonable?**
All columns fall within plausible physiological ranges:
- Age 20–79, Height 123–222 cm, Weight 36–132 kg
- Duration 1–30 min, Heart_Rate 67–128 bpm, Body_Temp 37.1–41.5 °C
- Calories 1–314 kcal

No negative, zero, or clearly impossible values at first glance.

In [10]:
def check_key(df: pd.DataFrame, name: str, key: str = "User_ID") -> None:
    print(f"{name}: rows={len(df)}, unique {key}={df[key].nunique()}, is_unique={df[key].is_unique}")

check_key(exercise, "exercise")
check_key(calories, "calories")

exercise: rows=15000, unique User_ID=15000, is_unique=True
calories: rows=15000, unique User_ID=15000, is_unique=True


In [11]:
exercise_ids = set(exercise['User_ID'])
calories_ids = set(calories['User_ID'])

only_in_exercise = exercise_ids - calories_ids
only_in_calories = calories_ids - exercise_ids

print("only in exercise:", len(only_in_exercise))
print("only in calories:", len(only_in_calories))
print("same ID sets:", exercise_ids == calories_ids)
print("Intersection size:", len(exercise_ids & calories_ids))

only in exercise: 0
only in calories: 0
same ID sets: True
Intersection size: 15000


## User_ID Validation

**Results:**
- exercise: 15000 rows, 15000 unique User_IDs → unique
- calories: 15000 rows, 15000 unique User_IDs → unique
- ID sets are identical: no ID exists in only one file.

**Implication for Merge:**
Since both keys are unique and the ID sets match exactly,
`inner` and `left` joins will produce the same number of rows (15000).
I will use `inner` for safety and clarity.

In [12]:
def audit_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate a data quality audit table for each column of the input DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to be audited.

    Returns
    -------
    pd.DataFrame
        A DataFrame indexed by column name with the following metrics:
        - dtype     : data type of the column
        - non_null  : number of non-missing values
        - missing   : number of missing values
        - missing_% : percentage of missing values (0–100), rounded to 2 decimals
        - n_unique  : number of unique values in the column
    """
    audit = pd.DataFrame({
        "dtype":     df.dtypes,
        "non_null":  df.notna().sum(),
        "missing":   df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique":  df.nunique(),
    })
    return audit

In [13]:
print("Shape:", exercise.shape)
print("duplicated rows:", exercise.duplicated().sum())
print()
audit_table(exercise)

Shape: (15000, 8)
duplicated rows: 0



,dtype,non_null,missing,missing_%,n_unique
User_ID,int64,15000,0,0.0,15000
Gender,str,15000,0,0.0,2
Age,int64,15000,0,0.0,60
Height,float64,15000,0,0.0,90
Weight,float64,15000,0,0.0,91
Duration,float64,15000,0,0.0,30
Heart_Rate,float64,15000,0,0.0,59
Body_Temp,float64,15000,0,0.0,45


In [14]:

print("Shape:", calories.shape)
print("duplicated rows:", calories.duplicated().sum())
print()
audit_table(calories)

Shape: (15000, 2)
duplicated rows: 0



,dtype,non_null,missing,missing_%,n_unique
User_ID,int64,15000,0,0.0,15000
Calories,float64,15000,0,0.0,277


## Data Audit Summary

**Missing values:** none in either dataset (0 across all columns).

**Duplicates:**
- exercise: 0 fully duplicated rows
- calories: 0 fully duplicated rows

**User_ID:** unique in both files (15000 = 15000). It is an identifier,
so it should NOT be used in numerical analysis (mean/median/correlation).

**Missing %:** all columns show 0.0% missing.

In [15]:
num_cols = ["Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]
print("values <= 0 per column:")
print((exercise[num_cols] <= 0).sum())
print("Calories <= 0:", (calories["Calories"] <= 0).sum())

print(exercise["Gender"].value_counts())

print("duplicate User_ID (exercise):", exercise.duplicated(subset="User_ID").sum())
print("duplicate User_ID (calories):", calories.duplicated(subset="User_ID").sum())

values <= 0 per column:
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
dtype: int64
Calories <= 0: 0
Gender
female    7553
male      7447
Name: count, dtype: int64
duplicate User_ID (exercise): 0
duplicate User_ID (calories): 0


## Missing, Duplicate & Hidden Missing

**Real missing values:** none.

**Hidden missing check (values that behave like missing):**
- Duration <= 0: 0
- Heart_Rate <= 0: 0
- Body_Temp <= 0: 0
- Age <= 0: 0
- Height <= 0: 0
- Weight <= 0: 0
- Calories <= 0: 0

**Duplicates:**
- Fully duplicated rows: 0 in both files
- Duplicated User_ID: 0 in both files

**Final decision:**
No missing values and no duplicates were found.
No rows were dropped and no imputation was applied.
The data is clean and ready for outlier analysis.

In [16]:
df = exercise.merge(calories,on="User_ID",how="inner",validate="one_to_one")
df.head()

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [17]:
print("exercise shape:", exercise.shape)
print("calories shape:", calories.shape)
print("merged shape:  ", df.shape)
print()
print("row count preserved?", len(df) == len(exercise))
print("calories has NaN?", df["Calories"].isna().sum())
print("id unique in merged?", df["User_ID"].is_unique)

exercise shape: (15000, 8)
calories shape: (15000, 2)
merged shape:   (15000, 9)

row count preserved? True
calories has NaN? 0
id unique in merged? True


In [18]:
test_merge = exercise.merge(calories,on="User_ID",how="outer",indicator=True)
print(test_merge['_merge'].value_counts())

_merge
both          15000
left_only         0
right_only        0
Name: count, dtype: int64


In [19]:
print("merged shape:", df.shape)
print("duplicated rows:", df.duplicated().sum())
print("duplicated User id:", df.duplicated(subset="User_ID").sum())
print()
audit_table(df)

merged shape: (15000, 9)
duplicated rows: 0
duplicated User id: 0



,dtype,non_null,missing,missing_%,n_unique
User_ID,int64,15000,0,0.0,15000
Gender,str,15000,0,0.0,2
Age,int64,15000,0,0.0,60
Height,float64,15000,0,0.0,90
Weight,float64,15000,0,0.0,91
Duration,float64,15000,0,0.0,30
Heart_Rate,float64,15000,0,0.0,59
Body_Temp,float64,15000,0,0.0,45
Calories,float64,15000,0,0.0,277


## Merge Decision

**Merge type:** `inner`

**Why inner?**
- Both files share the same User_ID set (verified in step 2).
- `inner` and `left` would produce identical results here.
- `inner` is chosen because it guarantees we only keep rows with both exercise and calorie data.

**Why `validate="one_to_one"`?**
- It raises an error if the key is duplicated on either side.
- A duplicated key in a merge causes a cartesian product:
  rows multiply (e.g. 1 duplicated ID × 2 matches = 2 rows instead of 1).
- With `one_to_one`, we catch this bug immediately instead of silently
  inflating the dataset.

**Result:**
- exercise: 15000 × 8
- calories: 15000 × 2
- merged:   15000 × 9  →  row count preserved